# 00 — Data Loader & Semi-Supervised Simulation

Este notebook es el punto de entrada de datos para todo el proyecto.
Carga el dataset limpio que dejó Marines y define la función `make_semi_supervised`
que oculta etiquetas para simular el escenario semi-supervisado.

**Uso por otros notebooks:**

```python
%run 00_data_loader.ipynb
X, y, feature_names, class_names = load_dataset()
X_train, X_test, y_train_partial, y_train_full, y_test = make_semi_supervised_split(
    X, y, label_fraction=0.10, test_size=0.30, random_state=42
)
```

In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

# Resolver paths de forma robusta independientemente del cwd
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = REPO_ROOT / 'data' / 'processed' / 'dataset_clean.csv'
MAPPING_PATH = REPO_ROOT / 'data' / 'processed' / 'class_mapping.json'

assert DATA_PATH.exists(), f"No se encontró el dataset en {DATA_PATH}"
print(f"Repo root: {REPO_ROOT}")
print(f"Dataset: {DATA_PATH}")

Repo root: /home/escu/Documents/Universidad/Semestres/7moSemestre/mineriaDatos/Lab10-MD
Dataset: /home/escu/Documents/Universidad/Semestres/7moSemestre/mineriaDatos/Lab10-MD/data/processed/dataset_clean.csv


In [2]:
def load_dataset():
    """
    Carga el dataset Dry Bean ya preprocesado.

    Returns
    -------
    X : np.ndarray de shape (n_samples, n_features)
        Features ya estandarizadas (mean=0, std=1).
    y : np.ndarray de shape (n_samples,)
        Etiquetas enteras 0..6.
    feature_names : list[str]
        Nombres de las 16 features en el mismo orden que las columnas de X.
    class_names : dict[int, str]
        Mapeo de id de clase a nombre original (0='BARBUNYA', ..., 6='SIRA').
    """
    df = pd.read_csv(DATA_PATH)
    feature_names = [c for c in df.columns if c not in ('Class', 'Class_name')]
    X = df[feature_names].values
    y = df['Class'].values.astype(int)

    with open(MAPPING_PATH) as f:
        class_names = {int(k): v for k, v in json.load(f).items()}

    return X, y, feature_names, class_names


# Smoke test
X, y, feature_names, class_names = load_dataset()
print(f"X.shape = {X.shape}")
print(f"y.shape = {y.shape}, clases únicas = {np.unique(y)}")
print(f"features ({len(feature_names)}): {feature_names}")
print(f"class_names: {class_names}")

X.shape = (13543, 16)
y.shape = (13543,), clases únicas = [0 1 2 3 4 5 6]
features (16): ['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4']
class_names: {0: 'BARBUNYA', 1: 'BOMBAY', 2: 'CALI', 3: 'DERMASON', 4: 'HOROZ', 5: 'SEKER', 6: 'SIRA'}


In [3]:
def make_semi_supervised(y, label_fraction, random_state=42, stratify=True):
    """
    Simula el escenario semi-supervisado: oculta etiquetas dejando solo
    `label_fraction` de ellas visibles. Las ocultas se marcan con -1.

    Convencion: -1 = "unlabeled" (compatible con sklearn.semi_supervised).

    Parameters
    ----------
    y : array de shape (n,)
        Etiquetas verdaderas.
    label_fraction : float en (0, 1]
        Proporcion de etiquetas que quedan visibles.
    random_state : int
        Semilla para reproducibilidad.
    stratify : bool
        Si True, mantiene la proporcion de clases entre las etiquetas
        visibles (recomendado para clases desbalanceadas).

    Returns
    -------
    y_partial : np.ndarray, copia de y con -1 en posiciones ocultas.
    labeled_mask : np.ndarray bool, True donde la etiqueta es visible.
    """
    rng = np.random.default_rng(random_state)
    n = len(y)
    labeled_mask = np.zeros(n, dtype=bool)

    if stratify:
        for c in np.unique(y):
            idx_c = np.where(y == c)[0]
            n_keep = max(1, int(round(label_fraction * len(idx_c))))
            chosen = rng.choice(idx_c, size=n_keep, replace=False)
            labeled_mask[chosen] = True
    else:
        n_keep = max(1, int(round(label_fraction * n)))
        chosen = rng.choice(n, size=n_keep, replace=False)
        labeled_mask[chosen] = True

    y_partial = y.copy()
    y_partial[~labeled_mask] = -1
    return y_partial, labeled_mask


# Smoke test
y_partial, mask = make_semi_supervised(y, label_fraction=0.10, random_state=0)
print(f"Etiquetas totales: {len(y)}")
print(f"Etiquetas visibles: {mask.sum()} ({mask.mean():.1%})")
print(f"Etiquetas ocultas:  {(~mask).sum()}")
print(f"Distribucion de clases visibles:\n{pd.Series(y_partial[mask]).value_counts().sort_index()}")

Etiquetas totales: 13543
Etiquetas visibles: 1355 (10.0%)
Etiquetas ocultas:  12188
Distribucion de clases visibles:
0    132
1     52
2    163
3    355
4    186
5    203
6    264
Name: count, dtype: int64


In [4]:
def make_semi_supervised_split(X, y, label_fraction, test_size=0.30, random_state=42):
    """
    Hace un split train/test estratificado y simula semi-supervisado SOLO en train.

    El test queda siempre con todas sus etiquetas (es el ground truth para evaluar).

    Returns
    -------
    X_train, X_test : np.ndarray
    y_train_partial : np.ndarray con -1 en las ocultas (input al modelo SS).
    y_train_full    : np.ndarray con todas las etiquetas reales (para analisis, NO se le pasa al modelo SS).
    y_test          : np.ndarray con etiquetas reales del test.
    """
    X_train, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    y_train_partial, _ = make_semi_supervised(
        y_train_full, label_fraction=label_fraction, random_state=random_state, stratify=True
    )
    return X_train, X_test, y_train_partial, y_train_full, y_test


# Smoke test
X_tr, X_te, y_tr_p, y_tr_full, y_te = make_semi_supervised_split(
    X, y, label_fraction=0.10, test_size=0.30, random_state=42
)
print(f"X_train: {X_tr.shape}, X_test: {X_te.shape}")
print(f"y_train etiquetadas: {(y_tr_p != -1).sum()} / {len(y_tr_p)}")
print(f"y_test (todas etiquetadas): {len(y_te)}")

X_train: (9480, 16), X_test: (4063, 16)
y_train etiquetadas: 947 / 9480
y_test (todas etiquetadas): 4063


## Contrato para los notebooks de modelos

Cualquier modelo del notebook 03 debe exponer una interfaz minima:

- `model.fit(X_train, y_train_partial)` — donde `y_train_partial` tiene `-1` en las muestras sin etiqueta.
- `model.predict(X_test) -> np.ndarray` — etiquetas predichas en 0..6.
- (Opcional, para analisis) `model.log_likelihood_history_` — lista de log-likelihoods por iteracion de EM.

El baseline supervisado debe **ignorar** las muestras con `y == -1` durante el `fit`
(filtrarlas antes de entrenar), para que la comparacion sea justa: ambos ven la misma
fraccion de etiquetas, pero el baseline solo puede usar las visibles.